-------------------------------
-------------------------------
# Laboratorio #8 - IA (CC3085)
* Dulce Ambrosio - 231143
* Daniel Chet - 231177
* Gadiel Ocaña - 231270

-------------------------------
-------------------------------

-------------------------------
**Task 2.1:** Implementación de Backtracking Search

-------------------------------

In [43]:
# -------------------------------
# Definición del problema (CSP)
# -------------------------------
# Slide 1 / Slide 3: El tema es CSP; aquí representamos el problema con:
## - Variables (máquinas)
## - Dominios (servidores posibles)
## - Restricciones (capacidad y anti-afinidad)

variables = ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8"]

domains = {
    var: ["S1", "S2", "S3"] for var in variables
}

# Restricciones de anti-afinidad (no pueden quedar en el mismo servidor)
constraints = [
    ("M1", "M2"),
    ("M3", "M4"),
    ("M5", "M6"),
    ("M1", "M5")
]

In [44]:
# Función para validar asignaciones
def is_valid(assignment, var, value):
    """
    Verifica consistencia al asignar 'value' a 'var'.
    Slide 2: Esto corresponde a la verificación de restricciones antes de aceptar una extensión.
    """

    # Copiamos asignación temporal (probando una extensión)
    temp = assignment.copy()
    temp[var] = value

    # -------------------------
    # Restricción 1: Capacidad
    # -------------------------
    # (CSP) Limita cuántas máquinas caben por servidor.
    count = {"S1": 0, "S2": 0, "S3": 0}

    for v in temp:
        count[temp[v]] += 1

    # Si algún servidor tiene más de 3 → inválido
    for server in count:
        if count[server] > 3:
            return False

    # -------------------------
    # Restricción 2: Anti-afinidad
    # -------------------------
    # Si un par (a,b) está asignado, deben quedar en servidores distintos.
    for (a, b) in constraints:
        if a in temp and b in temp:
            if temp[a] == temp[b]:
                return False

    return True

In [45]:
# Forward Checking
def forward_check(domains, assignment, var, value):
    """
    Reduce dominios futuros basado en la asignación actual (LOOKAHEAD).
    Slide 2: LOOKAHEAD/forward checking = actualizar dominios para podar temprano.
    """

    new_domains = {v: list(domains[v]) for v in domains}

    for (a, b) in constraints:
        # Si 'var' afecta a otra variable, eliminamos el valor prohibido en el vecino
        # (por anti-afinidad).
        if var == a and b not in assignment:
            if value in new_domains[b]:
                new_domains[b].remove(value)

        if var == b and a not in assignment:
            if value in new_domains[a]:
                new_domains[a].remove(value)

    return new_domains

In [46]:
# Backtracking Search
def backtracking(assignment, domains):
    """
    Algoritmo principal de Backtracking (DFS con poda).
    Slide 2: El patrón es 'elegir variable' -> 'probar valores' -> 'LOOKAHEAD' -> recursión.
    Slide 4: Conceptualmente explora un árbol completo (DFS), retrocediendo al fallar.
    """

    # -------------------------
    # Caso base: solución completa
    # -------------------------
    if len(assignment) == len(variables):
        return assignment

    # -------------------------
    # Seleccionar siguiente variable (simple)
    # -------------------------
    # Slide 2: aquí iría la heurística de selección de variable; usamos la primera no asignada.
    unassigned = [v for v in variables if v not in assignment]
    var = unassigned[0]

    # -------------------------
    # Probar valores
    # -------------------------
    # Slide 2: aquí también podríamos 'ordenar valores', usamos el orden del dominio.
    for value in domains[var]:

        if is_valid(assignment, var, value):

            # Slide 2: LOOKAHEAD mediante forward checking
            new_domains = forward_check(domains, assignment, var, value)

            # Si algún dominio queda vacío → podar (fail-fast)
            if any(len(new_domains[v]) == 0 for v in new_domains if v not in assignment):
                continue

            # Continuar recursión (DFS)
            result = backtracking(
                {**assignment, var: value},
                new_domains
            )

            if result is not None:
                return result

    return None

In [47]:
# Ejecución del algoritmo
 # Slide 2 / Slide 4: ejecutamos Backtracking (DFS + poda por restricciones/LOOKAHEAD).

solution = backtracking({}, domains)

print("Solución encontrada:")
print(solution)

Solución encontrada:
{'M1': 'S1', 'M2': 'S2', 'M3': 'S1', 'M4': 'S2', 'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}


-------------------------------
**Task 2.2:** Implementación de Beam Search

-------------------------------

In [48]:
# Función de evaluación (peso)

def compute_weight(assignment):
    """
    Mientras menos violaciones tenga una asignación, mejor es.
    Slide 6: en Greedy/Beam Search se elige la extensión con mayor 'peso' (heurística).
    """

    violations = 0

    # -------------------------
    # Restricción de capacidad
    # -------------------------
    count = {"S1": 0, "S2": 0, "S3": 0}

    for v in assignment:
        count[assignment[v]] += 1

    for server in count:
        if count[server] > 3:
            violations += (count[server] - 3)

    # -------------------------
    # Anti-afinidad
    # -------------------------
    for (a, b) in constraints:
        if a in assignment and b in assignment:
            if assignment[a] == assignment[b]:
                violations += 1

    # Queremos minimizar violaciones a peso negativo (más alto = mejor)
    return -violations

In [49]:
# Implementación de Beam Search

def beam_search(K):
    """
    Beam Search con tamaño K.
    Slide 7: Beam Search mantiene K candidatos simultáneamente.
    Slide 8: Mantener lista C (beam) de hasta K candidatos; EXTEND y luego PRUNE.
    Slide 9: Tradeoff: K=1 ≈ Greedy (Slide 5/6).
    """

    # Inicialización (lista C de candidatos parciales)
    beam = [{}]

    for var in variables:

        candidates = []

        # -------------------------
        # EXTEND
        # -------------------------
        # Slide 8: extender TODOS los candidatos actuales en el beam.
        for assignment in beam:
            for value in domains[var]:

                new_assignment = assignment.copy()
                new_assignment[var] = value

                candidates.append(new_assignment)

        # -------------------------
        # PRUNE (quedarse con los mejores K)
        # -------------------------
        # Slide 8: ordenar por 'peso' y conservar solo los K mejores.
        candidates.sort(key=lambda x: compute_weight(x), reverse=True)

        beam = candidates[:K]

    # -------------------------
    # Elegir mejor solución final
    # -------------------------
    best = max(beam, key=lambda x: compute_weight(x))

    return best

In [50]:
# Ejecutar el algoritmo

# Slide 8: ejecutamos Beam Search con un beam (C) de tamaño K.
solution = beam_search(K=3)

print("Solución encontrada con Beam Search:")
print(solution)
print("Peso:", compute_weight(solution))

Solución encontrada con Beam Search:
{'M1': 'S1', 'M2': 'S2', 'M3': 'S1', 'M4': 'S2', 'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}
Peso: 0


-------------------------------
**Task 2.3:** Implementación de Local Search (ICM) 

-------------------------------

In [51]:
# Inicialización aleatoria

import random

def random_assignment():
    """
    Crea una asignación completa aleatoria.
    Slide 15: ICM inicia desde una asignación aleatoria antes de mejorarla.
    """
    return {var: random.choice(domains[var]) for var in variables}

In [52]:
# Implementación de ICM

def local_search_icm(max_iters=100):
    """
    Implementación de ICM (Iterated Conditional Modes).
    Slide 10: Transición a 'Local Search'.
    Slide 11: En Local Search se modifican asignaciones COMPLETAS, no parciales.
    Slide 15: ICM recorre variable por variable escogiendo el valor que maximiza el peso.
    Slide 16: El peso no disminuye; converge en tiempo finito (puede caer en óptimos locales).
    """

    # -------------------------
    # Inicialización aleatoria
    # -------------------------
    # Slide 15: punto de partida aleatorio.
    current = random_assignment()

    for iteration in range(max_iters):

        improved = False

        # -------------------------
        # Iterar sobre variables
        # -------------------------
        for var in variables:

            best_value = current[var]
            best_weight = compute_weight(current)

            # -------------------------
            # Probar todos los valores
            # -------------------------
            # Slide 13: 'pasos pequeños' = cambiar UNA variable y ver si mejora el peso.
            for value in domains[var]:

                temp = current.copy()
                temp[var] = value

                weight = compute_weight(temp)

                if weight > best_weight:
                    best_weight = weight
                    best_value = value

            # -------------------------
            # Actualizar si mejora
            # -------------------------
            if best_value != current[var]:
                current[var] = best_value
                improved = True
                # Slide 16: al aceptar solo mejoras, el peso aumenta o se mantiene.

        # -------------------------
        # Si no hubo mejora → convergió
        # -------------------------
        if not improved:
            break

    return current

In [53]:
# Ejecutar el algoritmo

# Slide 15: correr ICM desde una asignación aleatoria hasta convergencia (o max_iters).
solution = local_search_icm(max_iters=200)

print("Solución encontrada con ICM:")
print(solution)
print("Peso:", compute_weight(solution))


Solución encontrada con ICM:
{'M1': 'S2', 'M2': 'S3', 'M3': 'S3', 'M4': 'S2', 'M5': 'S3', 'M6': 'S1', 'M7': 'S2', 'M8': 'S1'}
Peso: 0


-------------------------------
**Task 2.4:** Benchmarking y Conclusiones

-------------------------------

In [54]:
# Se miden los tiempos de ejecución de cada algoritmo

import time

# -------------------------
# Función para validar solución final
# -------------------------
def is_solution_valid(solution):
    """
    Valida una asignación completa contra las restricciones.
    (Útil para comparar enfoques exactos vs aproximados).
    Slide 17: Backtracking es exacto; Beam/ICM son aproximados.
    """
    if solution is None:
        return False

    # Capacidad
    count = {"S1": 0, "S2": 0, "S3": 0}
    for v in solution:
        count[solution[v]] += 1

    for server in count:
        if count[server] > 3:
            return False

    # Anti-afinidad
    for (a, b) in constraints:
        if solution[a] == solution[b]:
            return False

    return True


# -------------------------
# Backtracking
# -------------------------
# Slide 17: algoritmo exacto (complejidad exponencial en general).
start = time.time()
bt_solution = backtracking({}, domains)
bt_time = time.time() - start

# -------------------------
# Beam Search
# -------------------------
# Slide 17: algoritmo aproximado; su costo crece con K y el tamaño del problema.
start = time.time()
beam_solution = beam_search(K=3)
beam_time = time.time() - start

# -------------------------
# Local Search (ICM)
# -------------------------
# Slide 17: algoritmo aproximado; típicamente lineal en iteraciones*variables*dominio.
start = time.time()
icm_solution = local_search_icm(max_iters=200)
icm_time = time.time() - start


# -------------------------
# Resultados
# -------------------------
print("===== RESULTADOS =====\n")

print("Backtracking:")
print("Solución:", bt_solution)
print("Válida:", is_solution_valid(bt_solution))
print("Tiempo:", bt_time, "\n")

print("Beam Search:")
print("Solución:", beam_solution)
print("Válida:", is_solution_valid(beam_solution))
print("Peso:", compute_weight(beam_solution))
print("Tiempo:", beam_time, "\n")

print("Local Search (ICM):")
print("Solución:", icm_solution)
print("Válida:", is_solution_valid(icm_solution))
print("Peso:", compute_weight(icm_solution))
print("Tiempo:", icm_time)

===== RESULTADOS =====

Backtracking:
Solución: {'M1': 'S1', 'M2': 'S2', 'M3': 'S1', 'M4': 'S2', 'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}
Válida: True
Tiempo: 0.00020313262939453125 

Beam Search:
Solución: {'M1': 'S1', 'M2': 'S2', 'M3': 'S1', 'M4': 'S2', 'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}
Válida: True
Peso: 0
Tiempo: 0.0007081031799316406 

Local Search (ICM):
Solución: {'M1': 'S1', 'M2': 'S2', 'M3': 'S2', 'M4': 'S3', 'M5': 'S3', 'M6': 'S1', 'M7': 'S3', 'M8': 'S1'}
Válida: True
Peso: 0
Tiempo: 0.0008990764617919922
